## 1. Detect zipped or automatically unzipped Kaggle input


In [ ]:
from pathlib import Path
import base64, json, os, subprocess, sys, time, zipfile
INPUT_ROOT = Path(os.environ.get("JCAM_INPUT_ROOT", "/kaggle/input"))
WORK_ROOT = Path(os.environ.get("JCAM_WORK_ROOT", "/kaggle/working"))
RUN_ROOT = WORK_ROOT / "jcam_thm11_profile_class"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
RUN_OK = True
STAGES = []
def record(stage, outcome, detail):
    STAGES.append({"stage": stage, "outcome": outcome, "detail": detail})
    print(stage, outcome, detail)
mode = "embedded_exact_sources"
if INPUT_ROOT.exists():
    if any(INPUT_ROOT.rglob("check_profile_class_feedback.py")):
        mode = "kaggle_auto_unzipped_detected; embedded exact sources selected"
    elif any(INPUT_ROOT.rglob("*.zip")):
        mode = "kaggle_zip_detected; embedded exact sources selected"
record("bootstrap", "passed", mode)


## 2. Restore exact proof, experiment, and draft sources


In [ ]:
SOURCE_ARCHIVE = base64.b64decode("".join([
    '...'
]))
(RUN_ROOT / "source_package.zip").write_bytes(SOURCE_ARCHIVE)
with zipfile.ZipFile(RUN_ROOT / "source_package.zip") as archive:
    for member in archive.infolist():
        target = (RUN_ROOT / member.filename).resolve()
        if not target.is_relative_to(RUN_ROOT.resolve()):
            raise RuntimeError(f"Unsafe ZIP member: {member.filename}")
    archive.extractall(RUN_ROOT)
record("sources", "passed", "THM-11/PROP-03 proof, experiment, paper and audit")


## 3. Run direct Pareto-set coverage and interval-feedback checks


In [ ]:
started = time.perf_counter()
script = RUN_ROOT / "Code" / "experiments" / "check_profile_class_feedback.py"
result = subprocess.run([sys.executable, str(script)], cwd=RUN_ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(result.stdout[-6000:])
if result.returncode:
    RUN_OK = False
    record("coverage_check", "failed", f"exit={result.returncode}")
else:
    summary_path = RUN_ROOT / "Code" / "results" / "profile_class_feedback" / "profile_class_feedback_summary.json"
    summary = json.loads(summary_path.read_text())
    RUN_OK = bool(summary["all_checks_passed"] and summary["rows"] == 64 and summary["endpoint_prior_grid_checks"] == 1936 and summary["quadratic_identity_checks"] == 315)
    record("coverage_check", "passed" if RUN_OK else "failed", f"rows={summary['rows']}; prior_pairs={summary['endpoint_prior_grid_checks']}; seconds={time.perf_counter()-started:.2f}")


## 4. Zip all outputs and exact sources


In [ ]:
(RUN_ROOT / "notebook_stages.json").write_text(json.dumps({"run_ok": RUN_OK, "stages": STAGES}, indent=2), encoding="utf-8")
FINAL_ZIP = WORK_ROOT / "result_jcam_thm11_profile_class.zip"
with zipfile.ZipFile(FINAL_ZIP, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for path in sorted(RUN_ROOT.rglob("*")):
        if path.is_file() and path.name != "source_package.zip":
            archive.write(path, arcname=(Path("thm11_profile_class") / path.relative_to(RUN_ROOT)).as_posix())
print("Output archive:", FINAL_ZIP, "bytes:", FINAL_ZIP.stat().st_size, "run_ok:", RUN_OK)
